# Case study 1 · model discrimination with the HR criterion

Section 4.1 of the MIDDoE paper: Tabrizi Z., Barbera E., Leal da Silva W.R., Bezzo F. (2025). *MIDDoE: An MBDoE Python package for model identification, discrimination, and calibration.* Digital Chemical Engineering 17, 100276. https://doi.org/10.1016/j.dche.2025.100276

**The process.** A fed-batch baker's-yeast fermentation. Two states are measured: biomass `y1` and substrate
`y2` (both g/L). Three things can be manipulated: the dilution rate `u1(t)` [1/h], the feed substrate
concentration `u2(t)` [g/L] and the initial biomass `w1 = y1(0)` [g/L].

**The question.** Four kinetic models compete (paper eqs. 18a–d, files `MI.py` … `MIV.py` in this folder).
MI plays the real process and generates the data. Can one designed experiment tell MI from its closest rival?

**The workflow (paper Fig. 4).**

| Step | What happens | MIDDoE call |
|---|---|---|
| 1 | Describe the controls, measurements and models | the dictionaries `system`, `models`, `insilicos` |
| 2 | Run two preliminary experiments in silico | `krnl_expera.expera` |
| 3 | Fit all four models, run the t-tests and the P-test (Table 4) | `iden_parmest.parmest`, `iden_uncert.uncert` |
| 4 | Keep the two plausible models, design one experiment that separates them | `des_md.mbdoe_md` with `'HR'` |
| 5 | Run that experiment, refit, test again (Table 5) | `expera`, `parmest`, `uncert` |
| 6 | Store the rounds, plot and report | `save_rounds`, `save_to_jac`, `run_postprocessing` |

With HR the paper places every sampling point after about 24 h, where MI and MII diverge most (Fig. 6), and MI wins with P = 99.9 %. The folder `../scenario_BFF` repeats the study with the BFF criterion; the two notebooks
differ only in `des_opt['md_ob']`.

**How to run.** Open this notebook from its own folder (Jupyter and PyCharm do this by default) and run all
cells in order. MIDDoE writes every output next to the notebook: `data.xlsx` (the in-silico experiments, one
sheet per experiment), `iden_results.jac` (all rounds) and the folders `modelling/`, `confidence/`, `design/`,
`post_processing/`. `krnl_expera` *appends* to `data.xlsx`, so delete the outputs before running again.
The in-silico noise is random, so the numbers differ slightly from the paper on every run; the conclusions
should not.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/middoe_paper/case_study_1_model_discrimination/scenario_HR/runner.ipynb)

**On Google Colab?** Run the next cell first: it installs MIDDoE and downloads this case's model file(s)
into the session (on your own PC it does nothing). Colab gives 2 CPU cores and ends idle sessions, so for a
first run lower `num_parallel_runs`, the design iterations and `nboot`.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s) of this case; does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    base = 'https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/middoe_paper/case_study_1_model_discrimination/scenario_HR/'
    for f in ['MI.py', 'MII.py', 'MIII.py', 'MIV.py']:
        urllib.request.urlretrieve(base + f, f)

## 1 · Parameters of the four models

`theta` holds the values of each model (MII–MIV are adapted from Chen and Asprey, 2003). MI's values are the
*true* parameters of the in-silico process. Each fit starts 20 % above these values and may move ±50 %
around them (paper: "a feasible parameter space of ±50 % for model calibration").

In [ ]:
from pathlib import Path

theta = {
    'MI':   [0.25, 0.25, 0.88, 0.09],             # Monod + maintenance (true model): mu_max [1/h], Ks [g/L], Yxs [g/g], m [1/h]
    'MII':  [0.24987, 0.00912, 0.94194, 0.10154], # Contois: mu_max [1/h], Kx [-], Yxs [g/g], m [1/h]
    'MIII': [0.02636, 0.68145, 0.03381],          # linear growth: a [L/(g h)], Yxs [g/g], m [1/h]
    'MIV':  [0.15270, 0.40575, 0.47529],          # Monod without maintenance: mu_max [1/h], Ks [g/L], Yxs [g/g]
}

## 2 · The `system` dictionary: design space and constraints (paper Table 3)

`system` holds everything that is not a model parameter.

- `tvi`: the time-variant controls. Each one is a profile made of `stps` steps (CVP): `'CPF'` holds each
  level constant, `'LPF'` ramps linearly between levels. `const` constrains the order of the levels (`'rel'`
  free, `'inc'` rising, `'dec'` falling). `offl` and `offt` are the smallest allowed change of level and
  the shortest allowed time between two switches, in the units of the control and of time.
- `tvo`: the measured outputs. `sp` is the number of samples per experiment, `offt` the shortest time
  between two samples, `samp_s` a group label (outputs with the same label are sampled at the same times:
  "synchronised" in Table 3), `samp_f` sampling times forced into every design, and `unc` the noise
  standard deviation.
- `tii`: controls that are fixed for the whole run (here the initial biomass).
- `t_s` is the time horizon, `t_r` the time grid, `t_d` a dead time kept free of switches and samples at
  both ends.

In [ ]:
system = {
    'tvi': {                     # time-variant controls
        'u1': {                  # dilution rate [1/h]
            'stps': 5, 'cvp': 'CPF', 'const': 'rel',  # 5 levels, constant pieces, free order (Table 3: 4 switches, relaxed)
            'min': 0.05, 'max': 0.2,                  # bounds [1/h]
            'offl': 0.01, 'offt': 0.5,                # min level change [1/h], min time between switches [h]
        },
        'u2': {                  # feed substrate concentration [g/L]
            'stps': 5, 'cvp': 'LPF', 'const': 'rel',  # 5 levels joined by linear ramps
            'min': 5, 'max': 35,
            'offl': 1, 'offt': 0.5,
        },
    },
    'tvo': {                     # measured outputs
        'y1': {'init': 0, 'meas': True,    # biomass [g/L]; y1(0) comes from tii 'y10'
               'sp': 10, 'offt': 0.5,      # 10 samples, at least 0.5 h apart
               'samp_s': 1, 'samp_f': [],  # same group as y2 -> synchronised; no forced times
               'unc': 0.08},               # noise: 8 % of the value (insilicos 'errt' = 'rel')
        'y2': {'init': 0.01, 'meas': True, # substrate [g/L], y2(0) = 0.01 g/L
               'sp': 10, 'offt': 0.5, 'samp_s': 1, 'samp_f': [], 'unc': 0.08},
    },
    'tii': {'y10': {'min': 1, 'max': 5.5}},  # initial biomass w1 [g/L]
    'tio': {},                               # no time-invariant outputs
    't_s': [0, 40],                          # 40-h batch
    't_r': 0.16,                             # time grid [h] for simulation, profiles and sampling
    't_d': 0,                                # no dead time
}

## 3 · The `models` dictionary

`'pys'` means each model is a plain Python file with a `solve_model(t, y0, tii, tvi, theta)` function.
`theta` is where each fit starts; `t_l` and `t_u` are absolute bounds.

In [ ]:
models = {
    'can_m': ['MI', 'MII', 'MIII', 'MIV'],                    # candidate models still in play
    'krt': {m: 'pys' for m in theta},                         # model type: standalone Python file
    'src': {m: str(Path(f'{m}.py').resolve()) for m in theta},  # the files in this folder
    'theta': {m: [x * 1.2 for x in v] for m, v in theta.items()},  # start 20 % above the nominal values
    't_l': {m: [x * 0.5 for x in v] for m, v in theta.items()},    # lower bounds: -50 %
    't_u': {m: [x * 1.5 for x in v] for m, v in theta.items()},    # upper bounds: +50 %
}

## 4 · Step 1–2 · two preliminary experiments

`insilicos` says which model plays the process, with which parameters and noise, and lists the
preliminary experiments: constant `u1` of 0.05 and 0.10 1/h, `u2` = 30 g/L (Section 4.1.1).

The paper text gives `w1 = 1.00 g/L`. The notebook behind the published tables passed it under a key that
MIDDoE did not read (`'y_0'`), so the upper bound 5.5 g/L was used instead; Fig. 5, where `y1` starts near
5.5 g/L, confirms it. 5.5 reproduces the published data; use 1.0 to follow the text.

Each `expera` call simulates one experiment, samples it `sp` times, adds noise and appends a sheet to
`data.xlsx`.

In [ ]:
insilicos = {
    'tr_m': 'MI',                 # model that plays the real process
    'theta': theta['MI'],         # its true parameters
    'errt': 'rel',                # noise relative to the value (sigma = unc * |y|)
    'prels': {                    # preliminary experiments, one sheet each
        '1': {'u1': 0.05, 'u2': 30, 'y10': 5.5},
        '2': {'u1': 0.10, 'u2': 30, 'y10': 5.5},
    },
}

from middoe.krnl_expera import expera
expera(system, models, insilicos, design_decisions={}, expr=1)
expera(system, models, insilicos, design_decisions={}, expr=2)

## 5 · The `iden_opt` dictionary: estimation and uncertainty

Weighted least squares, solved with the trust-region constrained method (`'TC'`, called `'SQP'` when the
paper was run; the paper text says DE, the notebook used trust-region). The parameter covariance comes from
the Hessian of the fit (`'H'`); it gives the 95 % confidence intervals and t-values.

In [ ]:
iden_opt = {
    'meth': 'TC',         # optimiser: 'TC' trust-region; also 'SLSQP', 'LMBFGS', 'NMS', 'BFGS', 'DE'
    'ms': False,          # no multi-start
    'maxit': 500, 'tol': 1e-2,  # iteration limit and tolerance of the paper-era trust-region run
    'ob': 'WLS',          # weighted least squares; also 'LS', 'MLE', 'Chi'
    'var-cov': 'H',       # covariance from the Hessian; 'M' Fisher matrix, 'B' bootstrap
    'sens_m': 'central',  # central finite differences for the sensitivities
    'eps': 1e-3,          # their step, in the normalised parameter space
    'c_plt': True,        # plot confidence ellipses
    'f_plt': True,        # plot fits against the data
    'plt_s': True,        # show the plots here as well as saving them
    'log': True,          # print the optimiser progress
}

## 6 · Step 3 · round 1: fit all four models (Table 4)

`parmest` fits every model in `can_m` to `data.xlsx`. `uncert` adds the covariance, confidence intervals,
t-values, R² and the P-test (the probability of each model being the right one), and writes the new estimates
back into `models['theta']`. `save_rounds` stores the round, draws the plots and prints the reference t-value.

Paper: MI 90.47 %, MII 9.53 %, MIII and MIV ≈ 0 %.

In [ ]:
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds

resultpr = parmest(system, models, iden_opt)
uncert_results = uncert(resultpr, system, models, iden_opt)
round_data = {}
save_rounds(1, uncert_results['results'], 'preliminary', round_data, models, iden_opt, uncert_results['obs'], system)

## 7 · Step 4 · design the discriminating experiment

MIII and MIV are dropped because their P-values are negligible. The design then maximises the divergence
between MI and MII with the chosen criterion: `'HR'` (Hunter–Reiner) weighs the squared difference of the
predictions, `'BFF'` (Buzzi-Ferraris–Forzatti) also accounts for the uncertainty of each model's predictions.

The decision variables are the levels and switching times of `u1` and `u2`, `w1` and the 10 sampling times,
all within the constraints of `system`. `'PS'` is pattern search (as in the paper runs; the paper text
describes DE followed by pattern search, `'DEPS'`). `mbdoe_md` starts 16 independent searches in parallel and
keeps the best design.

In [ ]:
models['can_m'].remove('MIII')
models['can_m'].remove('MIV')

des_opt = {
    'md_ob': 'HR',    # discrimination criterion: 'HR' or 'BFF'
    'meth': 'PS',        # optimiser: 'PS' pattern search, 'DE' differential evolution, 'DEPS' both
    'eps': 1e-3,         # finite-difference step for the sensitivities used by 'BFF'
    'plt': True,         # plot the designed experiment
    'itr': {'maxmd': 200, 'tolmd': 1, 'pps': 30},   # generations, tolerance, DE population size
}

from middoe.des_md import mbdoe_md
designs = mbdoe_md(des_opt, system, models, round=2, num_parallel_runs=16)

## 8 · Step 5 · round 2: run the design and test again (Table 5)

The designed experiment becomes sheet 3 of `data.xlsx`; MI and MII are refitted on all three experiments.
Paper: P(MI) = 99.9 %.

In [ ]:
expera(system, models, insilicos, designs, expr=3, swps=designs['swps'])

resultpr = parmest(system, models, iden_opt)
uncert_results = uncert(resultpr, system, models, iden_opt)
save_rounds(2, uncert_results['results'], 'MBDoE-MD', round_data, models, iden_opt, uncert_results['obs'], system)

## 9 · Step 6 · store and report

`save_to_jac` keeps all rounds in `iden_results.jac`. `run_postprocessing` plots the P- and t-values across
rounds and the confidence regions, and writes an Excel report per model and round to `post_processing/`.

In [ ]:
from middoe.log_utils import save_to_jac, load_from_jac
from middoe.iden_utils import run_postprocessing

save_to_jac(round_data, purpose='iden')
run_postprocessing(round_data=load_from_jac()['iden'], solvers=['MI', 'MII'], selected_rounds=[1, 2],
                   plot_global_p_and_t=True, plot_confidence_spaces=True, plot_p_and_t_tests=True,
                   export_excel_reports=True, plot_estimability=False)